# Colachel: Blender EEVEE render on Kaggle

1. **Settings** (cell 1): pick the `.blend`, frame range and resolution.
2. Run cells 2-4 once per session: GPU check, install Blender 5.2, **EEVEE headless test**.
3. Cell 4 prints which mode works (and that it uses the NVIDIA GPU, not a software renderer) and how many
   seconds a frame takes. **Compare `test_<mode>.png` with the same frame rendered on your PC** before trusting it.
4. Cell 5 renders the range with the working mode; cell 6 zips the frames for download.

Notebook settings (right panel): **Accelerator = GPU T4 x2 or P100**, **Internet = On**,
and add the dataset `colachel-blender` (Add Input).
For long renders use **Save Version → Save & Run All (Commit)**: it runs in the background up to 12 h and the
zip appears under the notebook's *Output* tab.

In [ ]:
# 1. SETTINGS
BLEND   = "Kochi"         # name of the .blend in the dataset (without .blend)
START   = 147             # first frame
END     = 368             # last frame
RES     = "1280x720"      # 1280x720 review, 1920x1080 final
SAMPLES = 24              # 24 review, 32 final
TEST_FRAME = 230          # frame used by the headless test (compare it with your PC's render)
BLENDER_VERSION = "5.2.2"

import glob, os
DATA = glob.glob("/kaggle/input/*colachel*")[0]
OUT  = f"/kaggle/working/frames_{BLEND}_{START}_{END}"
print("dataset:", DATA, os.listdir(DATA))

In [ ]:
# 2. GPU CHECK
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!ls /usr/share/vulkan/icd.d/ /etc/vulkan/icd.d/ 2>/dev/null; ldconfig -p | grep -E "libEGL_nvidia|libGLX_nvidia|libnvidia-glcore" | head

In [ ]:
# 3. INSTALL BLENDER 5.2 (+ libraries and a virtual display as a fallback)
!apt-get -qq update > /dev/null && apt-get -qq install -y xvfb libxi6 libxxf86vm1 libxfixes3 libxrender1 libxkbcommon0 libsm6 libgl1 libegl1 libvulkan1 > /dev/null
ver = BLENDER_VERSION
major = ".".join(ver.split(".")[:2])
url = f"https://download.blender.org/release/Blender{major}/blender-{ver}-linux-x64.tar.xz"
if not os.path.exists("/kaggle/working/blender/blender"):
    !wget -q {url} -O /tmp/blender.tar.xz && mkdir -p /kaggle/working/blender && tar -xf /tmp/blender.tar.xz -C /kaggle/working/blender --strip-components=1
BLENDER = "/kaggle/working/blender/blender"
!{BLENDER} --version | head -1

In [ ]:
# 4. EEVEE HEADLESS TEST: tries three ways, reports the GPU Blender really uses and the time per frame
import subprocess, time, re
probe = "import gpu; print('GPU_RENDERER', gpu.platform.renderer_get(), '|', gpu.platform.vendor_get(), '|', gpu.platform.backend_type_get())"
modes = {
    "background":  [BLENDER, "-b"],
    "vulkan":      [BLENDER, "-b", "--gpu-backend", "vulkan"],
    "xvfb":        ["xvfb-run", "-a", "-s", "-screen 0 1920x1080x24", BLENDER, "-b"],
}
results = {}
for name, cmd in modes.items():
    t = time.time()
    r = subprocess.run(cmd + [f"{DATA}/{BLEND}.blend", "--python-expr",
                       probe + f"; import bpy; bpy.context.scene.eevee.taa_render_samples={SAMPLES}",
                       "-P", f"{DATA}/render_cli.py", "--", "stills", f"/kaggle/working/test_{name}",
                       str(TEST_FRAME), "--res", RES], capture_output=True, text=True, timeout=900)
    dt = time.time() - t
    log = r.stdout + r.stderr
    gpu_line = next((l for l in log.splitlines() if "GPU_RENDERER" in l), "no GPU line")
    ok = os.path.exists(f"/kaggle/working/test_{name}_f{TEST_FRAME:03d}.png")
    results[name] = (ok, dt, gpu_line)
    print(f"{name:11s} ok={ok}  {dt:6.1f}s  {gpu_line}")
    if not ok:
        print("   last log lines:", "\n   ".join(log.strip().splitlines()[-6:]))

good = [n for n, (ok, dt, g) in results.items() if ok and re.search("NVIDIA|Tesla|T4|P100", g, re.I)]
MODE = min(good, key=lambda n: results[n][1]) if good else None
print("\n==> use mode:", MODE or "NONE (EEVEE cannot use the GPU here; see notes)")
from IPython.display import Image, display
for n, (ok, dt, g) in results.items():
    if ok:
        print(n); display(Image(f"/kaggle/working/test_{n}_f{TEST_FRAME:03d}.png", width=640))

In [ ]:
# 5. RENDER THE RANGE (resumable: re-run to continue; skips frames already done)
assert MODE, "No working GPU mode: do not render here (see cell 4)."
cmd = modes[MODE] + [f"{DATA}/{BLEND}.blend", "--python-expr",
                     f"import bpy; bpy.context.scene.eevee.taa_render_samples={SAMPLES}",
                     "-P", f"{DATA}/render_cli.py", "--", "frames", OUT,
                     "--range", str(START), str(END), "--res", RES, "--pause", "0"]
t = time.time()
p = subprocess.run(cmd, capture_output=True, text=True)
done = len(glob.glob(f"{OUT}/f_*.png"))
print(f"{done} frames in {OUT}  ({(time.time()-t)/max(1,done):.1f} s/frame)")
print("\n".join((p.stdout + p.stderr).strip().splitlines()[-5:]))

In [ ]:
# 6. ZIP FOR DOWNLOAD (Output tab, or the link below)
import shutil
for f in glob.glob(f"{OUT}/f_*.png"):
    if os.path.getsize(f) < 2048: os.remove(f)       # drop empty placeholders from interrupted frames
zipname = shutil.make_archive(f"/kaggle/working/{BLEND}_{START}_{END}", "zip", OUT)
print(zipname, round(os.path.getsize(zipname)/1e6, 1), "MB")
from IPython.display import FileLink; display(FileLink(os.path.relpath(zipname, "/kaggle/working")))